In [1]:
from dotenv import load_dotenv, find_dotenv
import os

_ = load_dotenv(find_dotenv(), override=True)
weaviate_api_key = os.getenv("WEAVIATE_API_KEY")
print(weaviate_api_key)

bUFyOForbUpXSW1jc0Z3ZF9XUkMybFRxVkxybWI4VThhYzY3bmNOUVpURUtwbVZYYWFwWmxZZzRHVHBJPV92MjAw


In [2]:
os.environ['WEAVIATE_API_KEY']

'bUFyOForbUpXSW1jc0Z3ZF9XUkMybFRxVkxybWI4VThhYzY3bmNOUVpURUtwbVZYYWFwWmxZZzRHVHBJPV92MjAw'

In [3]:
os.environ['WEAVIATE_URL']

'kljqjpzrxgrfav63lmogg.c0.us-west3.gcp.weaviate.cloud'

In [4]:
import weaviate
from weaviate.classes.init import Auth, AdditionalConfig, Timeout
auth_config = Auth.api_key(
    api_key=os.environ['WEAVIATE_API_KEY']
)

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ['WEAVIATE_URL'],
    auth_credentials=auth_config,
    additional_config=AdditionalConfig(
        timeout=Timeout(init=60, query=60, insert=60, stream=20)
    ),
    headers={
        "X-Cohere-Api-Key": os.environ["COHERE_API_KEY"]
    }
    # additional_headers={
    #     "X-OpenAI-Api-Key": os.environ['OPENAI_API_KEY']
    # })
    )

client.is_ready() 

True

In [6]:
import json

In [7]:
metainfo = client.get_meta()

print(json.dumps(metainfo, indent=2))

{
  "grpcMaxMessageSize": 104858000,
  "hostname": "http://[::]:8080",
  "modules": {
    "generative-anthropic": {
      "documentationHref": "https://docs.anthropic.com/en/api/getting-started",
      "name": "Generative Search - Anthropic"
    },
    "generative-anyscale": {
      "documentationHref": "https://docs.anyscale.com/endpoints/overview",
      "name": "Generative Search - Anyscale"
    },
    "generative-aws": {
      "documentationHref": "https://docs.aws.amazon.com/bedrock/latest/APIReference/welcome.html",
      "name": "Generative Search - AWS"
    },
    "generative-cohere": {
      "documentationHref": "https://docs.cohere.com/reference/chat",
      "name": "Generative Search - Cohere"
    },
    "generative-contextualai": {
      "documentationHref": "https://docs.contextual.ai/api-reference/generate/generate",
      "name": "Generative Search - Contextual AI"
    },
    "generative-databricks": {
      "documentationHref": "https://docs.databricks.com/en/machine-le

In [8]:
assert client.is_ready()

In [41]:
client.connect()

In [42]:
client.collections.delete("Movies")

In [43]:
from weaviate.classes.config import Configure, DataType, Property

client.collections.create(
    name="Movies",
    properties=[
        Property(name="title", data_type=DataType.TEXT),
        Property(name="overview", data_type=DataType.TEXT),
        Property(name="vote_average", data_type=DataType.NUMBER),
        Property(name="genre_ids", data_type=DataType.INT_ARRAY),
        Property(name="release_date", data_type=DataType.DATE),
        Property(name="tmdb_id", data_type=DataType.INT),
        Property(name="original_language", data_type=DataType.TEXT),
        Property(name="popularity", data_type=DataType.NUMBER),
        Property(name="vote_count", data_type=DataType.INT)
    ],
    # Define the vectorizer module
    vector_config=Configure.Vectors.text2vec_cohere(model="embed-v4.0"),
    # Define the generative module
    generative_config=Configure.Generative.cohere(model="command-a-03-2025")
)

client.close()

## Import Data 

In [44]:
import pandas as pd
import requests
from datetime import datetime, timezone
import json
from weaviate.util import generate_uuid5
from tqdm import tqdm
import os

data_url = "https://raw.githubusercontent.com/weaviate-tutorials/edu-datasets/main/movies_data_1990_2024.json"
resp = requests.get(data_url)

df = pd.DataFrame(resp.json())
df.head()

,backdrop_path,genre_ids,id,original_language,original_title,overview,popularity,poster_path,release_date,title,video,vote_average,vote_count
0,/3Nn5BOM1EVw1IYrv6MsbOS6N1Ol.jpg,"[14, 18, 10749]",162,en,Edward Scissorhands,A small suburban town receives a visit from a ...,45.694,/1RFIbuW9Z3eN9Oxw2KaQG5DfLmD.jpg,1990-12-07,Edward Scissorhands,False,7.7,12305
1,/sw7mordbZxgITU877yTpZCud90M.jpg,"[18, 80]",769,en,GoodFellas,"The true story of Henry Hill, a half-Irish, ha...",57.228,/aKuFiU82s5ISJpGZp7YkIr3kCUd.jpg,1990-09-12,GoodFellas,False,8.5,12106
2,/6uLhSLXzB1ooJ3522ydrBZ2Hh0W.jpg,"[35, 10751]",771,en,Home Alone,Eight-year-old Kevin McCallister makes the mos...,3.538,/onTSipZ8R3bliBdKfPtsDuHTdlL.jpg,1990-11-16,Home Alone,False,7.4,10599
3,/vKp3NvqBkcjHkCHSGi6EbcP7g4J.jpg,"[12, 35, 878]",196,en,Back to the Future Part III,The final installment of the Back to the Futur...,28.896,/crzoVQnMzIrRfHtQw0tLBirNfVg.jpg,1990-05-25,Back to the Future Part III,False,7.5,9918
4,/3tuWpnCTe14zZZPt6sI1W9ByOXx.jpg,"[35, 10749]",114,en,Pretty Woman,When a millionaire wheeler-dealer enters a bus...,97.953,/hVHUfT801LQATGd26VPzhorIYza.jpg,1990-03-23,Pretty Woman,False,7.5,7671


In [45]:
df.columns

Index(['backdrop_path', 'genre_ids', 'id', 'original_language',
       'original_title', 'overview', 'popularity', 'poster_path',
       'release_date', 'title', 'video', 'vote_average', 'vote_count'],
      dtype='str')

In [51]:
## Configure the collection object

movies = client.collections.use("Movies")

In [52]:
df_test = pd.DataFrame([[1, 1.5], [2, 3.0], [3, 4.5]], columns=["int", "float"])
df_test

,int,float
0,1,1.5
1,2,3.0
2,3,4.5


In [53]:
for index, series_ in df_test.iterrows():
    print(series_['int'])

1.0
2.0
3.0


In [54]:
client.connect()

In [55]:
with movies.batch.fixed_size(batch_size=200) as batch:
    for i, movie in tqdm(df.iterrows()):
        release_date = datetime.fromisoformat(movie["release_date"]).replace(tzinfo=timezone.utc)
        genre_ids = json.loads(movie["genre_ids"])

        # Build the object payload
        movie_obj = {
            "title": movie["title"],
            "overview": movie["overview"],
            "vote_average": movie["vote_average"],
            "genre_ids": genre_ids,
            "release_date": release_date,
            "tmdb_id": movie["id"],
            "original_language": movie["original_language"],
            "popularity": movie['popularity'],
            "vote_count": movie['vote_count']
        }

        # Add object to batch queue
        batch.add_object(
            properties=movie_obj,
            uuid=generate_uuid5(movie["id"])
        )
        # Batcher automatically sends batches

# check for failed objects
if len(movies.batch.failed_objects) > 0:
    print(f"Failed to import {len(movies.batch.failed_objects)} objects")

client.close()  

680it [00:00, 1216.67it/s]


In [27]:
client.connect()

In [ ]:
collection = client.collections.get("Movies")
response =  collection.query.bm25(
    query="dystopian future",
    limit=3
)

print(response.objects)

[Object(uuid=_WeaviateUUIDInt('bb18affe-9ffe-56f5-babc-0c6c01257fd9'), metadata=MetadataReturn(creation_time=None, last_update_time=None, distance=None, certainty=None, score=None, explain_score=None, is_consistent=None, rerank_score=None), properties={'genre_ids': [28, 12, 878], 'vote_average': 7.5, 'tmdb_id': 127585, 'overview': 'The ultimate X-Men ensemble fights a war for the survival of the species across two time periods as they join forces with their younger selves in an epic battle that must change the past – to save our future.', 'title': 'X-Men: Days of Future Past', 'release_date': datetime.datetime(2014, 5, 15, 0, 0, tzinfo=datetime.timezone.utc)}, references=None, vector={}, collection='Movies'), Object(uuid=_WeaviateUUIDInt('1b510513-ebf3-5fb3-94ca-55cdb64a1300'), metadata=MetadataReturn(creation_time=None, last_update_time=None, distance=None, certainty=None, score=None, explain_score=None, is_consistent=None, rerank_score=None), properties={'genre_ids': [12, 35, 878], '

In [29]:
for obj in response.objects:
    print(obj.properties)

{'genre_ids': [28, 12, 878], 'vote_average': 7.5, 'tmdb_id': 127585, 'overview': 'The ultimate X-Men ensemble fights a war for the survival of the species across two time periods as they join forces with their younger selves in an epic battle that must change the past – to save our future.', 'title': 'X-Men: Days of Future Past', 'release_date': datetime.datetime(2014, 5, 15, 0, 0, tzinfo=datetime.timezone.utc)}
{'genre_ids': [12, 35, 878], 'vote_average': 7.5, 'tmdb_id': 196, 'overview': 'The final installment of the Back to the Future trilogy finds Marty digging the trusty DeLorean out of a mineshaft and looking for Doc in the Wild West of 1885. But when their time machine breaks down, the travelers are stranded in a land of spurs. More problems arise when Doc falls for pretty schoolteacher Clara Clayton, and Marty tangles with Buford Tannen.', 'title': 'Back to the Future Part III', 'release_date': datetime.datetime(1990, 5, 25, 0, 0, tzinfo=datetime.timezone.utc)}
{'genre_ids': [12

In [30]:
from weaviate.classes.query import Filter, MetadataQuery

# Instantiate your client (not shown). e.g.:
# client = weaviate.connect_to_weaviate_cloud(...) or
# client = weaviate.connect_to_local(...)

# Configure collection object
movies = client.collections.use("Movies")

# Perform query
response = movies.query.near_text(
    query="dystopian future",
    limit=5,
    return_metadata=MetadataQuery(distance=True)
)

# Inspect the response
for o in response.objects:
    print(o.properties["title"], o.properties["release_date"].year)  # Print the title and release year (note the release date is a datetime object)
    print(f"Distance to query: {o.metadata.distance:.3f}\n")  # Print the distance of the object from the query

client.close()

Gattaca 1997
Distance to query: 0.587

In Time 2011
Distance to query: 0.587

Her 2013
Distance to query: 0.605

I, Robot 2004
Distance to query: 0.613

Looper 2012
Distance to query: 0.616



In [32]:
from weaviate.classes.query import Filter, MetadataQuery
client.connect()

# Instantiate your client (not shown). e.g.:
# client = weaviate.connect_to_weaviate_cloud(...) or
# client = weaviate.connect_to_local(...)

# Configure collection object
movies = client.collections.use("Movies")

# Perform query
response = movies.query.bm25(
    query="history", limit=5, return_metadata=MetadataQuery(score=True)
)

# Inspect the response
for o in response.objects:
    print(o.properties["title"], o.properties["release_date"].year)  # Print the title and release year (note the release date is a datetime object)
    print(f"BM25 score: {o.metadata.score:.3f}\n")  # Print the BM25 score of the object from the query

client.close()

A Beautiful Mind 2001
BM25 score: 2.723

Legends of the Fall 1994
BM25 score: 2.483

Night at the Museum 2006
BM25 score: 2.412

Hacksaw Ridge 2016
BM25 score: 2.367

The Butterfly Effect 2004
BM25 score: 2.202



In [33]:
client.connect()
from weaviate.classes.query import Filter, MetadataQuery

# Configure collection object
movies = client.collections.use("Movies")

# Perform query
response = movies.query.hybrid(
    query="history", limit=5, return_metadata=MetadataQuery(score=True)
)

# Inspect the response
for o in response.objects:
    print(o.properties["title"], o.properties["release_date"].year)  # Print the title and release year (note the release date is a datetime object)
    print(f"Hybrid score: {o.metadata.score:.3f}\n")  # Print the hybrid search score of the object from the query

client.close()

Legends of the Fall 1994
Hybrid score: 0.936

The Butterfly Effect 2004
Hybrid score: 0.732

American History X 1998
Hybrid score: 0.706

Captain Marvel 2019
Hybrid score: 0.676

A Beautiful Mind 2001
Hybrid score: 0.568



### Filtering

In [34]:
client.connect()

In [35]:
movies = client.collections.use("Movies")

response = movies.query.near_text(
    query="dystopian future",
    limit=5,
    return_metadata=MetadataQuery(distance=True),
    filters=Filter.by_property("release_date").greater_than(datetime(2020, 1, 1))
)

# Inspect the response
for o in response.objects:
    print(o.properties["title"], o.properties["release_date"].year)  # Print the title and release year (note the release date is a datetime object)
    print(f"Distance to query: {o.metadata.distance:.3f}\n")  # Print the distance of the object from the query

client.close()

c:\Users\user\Documents\DeepLearning-Indepth\.venv\Lib\site-packages\weaviate\warnings.py:256: UserWarning: Con002: You are using the datetime object 2020-01-01 00:00:00 without a timezone. The timezone will be set to UTC.
            To use a different timezone, specify it in the datetime object. For example:
            datetime.datetime(2021, 1, 1, 0, 0, 0, tzinfo=datetime.timezone(-datetime.timedelta(hours=2))).isoformat() = 2021-01-01T00:00:00-02:00
            
  warnings.warn(


Dune 2021
Distance to query: 0.663

Jurassic World Dominion 2022
Distance to query: 0.687

The Flash 2023
Distance to query: 0.702

Greenland 2020
Distance to query: 0.715

The Adam Project 2022
Distance to query: 0.725



In [37]:
client.connect()

In [38]:
movies = client.collections.use("Movies")

response = movies.query.near_text(
    query="dystopian future",
    limit=5,
    return_metadata=MetadataQuery(distance=True),
    filters=Filter.by_property("release_date").greater_than(datetime(2020, 1, 1)),
    return_properties=['title', "overview", "release_date"]
)

print(response.objects)

# Inspect the response
for o in response.objects:
    print(o.properties["title"], o.properties["release_date"].year, o.properties['overview'])  # Print the title and release year (note the release date is a datetime object)
    print(f"Distance to query: {o.metadata.distance:.3f}\n")  # Print the distance of the object from the query

client.close()

c:\Users\user\Documents\DeepLearning-Indepth\.venv\Lib\site-packages\weaviate\warnings.py:256: UserWarning: Con002: You are using the datetime object 2020-01-01 00:00:00 without a timezone. The timezone will be set to UTC.
            To use a different timezone, specify it in the datetime object. For example:
            datetime.datetime(2021, 1, 1, 0, 0, 0, tzinfo=datetime.timezone(-datetime.timedelta(hours=2))).isoformat() = 2021-01-01T00:00:00-02:00
            
  warnings.warn(


[Object(uuid=_WeaviateUUIDInt('4dd16abf-3bd7-5427-8c75-2a6ac747a108'), metadata=MetadataReturn(creation_time=None, last_update_time=None, distance=0.6627321243286133, certainty=None, score=None, explain_score=None, is_consistent=None, rerank_score=None), properties={'overview': "Paul Atreides, a brilliant and gifted young man born into a great destiny beyond his understanding, must travel to the most dangerous planet in the universe to ensure the future of his family and his people. As malevolent forces explode into conflict over the planet's exclusive supply of the most precious resource in existence-a commodity capable of unlocking humanity's greatest potential-only those who can conquer their fear will survive.", 'title': 'Dune', 'release_date': datetime.datetime(2021, 9, 15, 0, 0, tzinfo=datetime.timezone.utc)}, references=None, vector={}, collection='Movies'), Object(uuid=_WeaviateUUIDInt('25fa19cb-a855-5c09-b5ca-bd4631571ed0'), metadata=MetadataReturn(creation_time=None, last_upd

In [39]:
from typing import List

In [58]:
client.connect()

In [59]:
def keyword_search(query, 
        results_lang = "en",
        properties: List[str] = ['title', "overview", "release_date", "vote_average"],
        num_results: int = 3):
    collection = client.collections.get("Movies")
    response = collection.query.bm25(
        query=query,
        filters=Filter.by_property("original_language").equal(results_lang),
        return_metadata=MetadataQuery(score=True),
        return_properties=properties,
        limit=num_results
    )
    result = [
        {
            **obj.properties,
            "score": obj.metadata.score
        }
        for obj in response.objects
    ]

    return result

keyword_search(query="Dystopia future")

[{'vote_average': 7.5,
  'overview': 'The ultimate X-Men ensemble fights a war for the survival of the species across two time periods as they join forces with their younger selves in an epic battle that must change the past – to save our future.',
  'title': 'X-Men: Days of Future Past',
  'release_date': datetime.datetime(2014, 5, 15, 0, 0, tzinfo=datetime.timezone.utc),
  'score': 3.363311767578125},
 {'vote_average': 7.5,
  'overview': 'The final installment of the Back to the Future trilogy finds Marty digging the trusty DeLorean out of a mineshaft and looking for Doc in the Wild West of 1885. But when their time machine breaks down, the travelers are stranded in a land of spurs. More problems arise when Doc falls for pretty schoolteacher Clara Clayton, and Marty tangles with Buford Tannen.',
  'title': 'Back to the Future Part III',
  'release_date': datetime.datetime(1990, 5, 25, 0, 0, tzinfo=datetime.timezone.utc),
  'score': 3.002270460128784},
 {'vote_average': 7.0,
  'overvi

In [60]:
import os
import weaviate

# Instantiate your client (not shown). e.g.:
# client = weaviate.connect_to_weaviate_cloud(...) or
# client = weaviate.connect_to_local(...)

# Configure collection object
movies = client.collections.use("Movies")

# Perform query
response = movies.generate.near_text(
    query="dystopian future",
    limit=5,
    single_prompt="Translate this into French: {title}"
)

# Inspect the response
for o in response.objects:
    print(o.properties["title"])  # Print the title
    print(o.generated)  # Print the generated text (the title, in French)

client.close()

In Time
"In Time" translates to "En temps" in French. However, if you're referring to the 2011 science fiction film "In Time" directed by Andrew Niccol, the French title is **"Time Out"**.
Her
The translation of "Her" into French depends on the context, as it can be a possessive adjective or a pronoun. Here are the possible translations:

1. **As a possessive adjective**:  
   - **Son** (when referring to a feminine singular noun, e.g., "her book" = "son livre").  
   - **Ses** (when referring to a plural noun, e.g., "her books" = "ses livres").  

2. **As a pronoun**:  
   - **Elle** (subject pronoun, e.g., "She is here" = "Elle est là").  
   - **La** (direct object pronoun, e.g., "I saw her" = "Je l'ai vue").  
   - **Lui** (indirect object pronoun, e.g., "I gave it to her" = "Je le lui ai donné").  

If you provide more context, I can give you the most accurate translation!
Gattaca
Le titre "Gattaca" reste inchangé en français, car il s'agit d'un nom propre et d'un titre de film qu

### 'Grouped task' generation
A 'grouped task' generation performs RAG queries on the set of retrieved objects. This is useful when you want to transform the set of objects as a whole, with one prompt.

Run this to find entries in "Movies" most similar to "dystopian future", find commonalities between them, and print out the results.

In [61]:
client.connect()

import os
import weaviate

# Instantiate your client (not shown). e.g.:
# client = weaviate.connect_to_weaviate_cloud(...) or
# client = weaviate.connect_to_local(...)

# Configure collection object
movies = client.collections.use("Movies")

# Perform query
response = movies.generate.near_text(
    query="dystopian future",
    limit=5,
    grouped_task="What do these movies have in common?",
    # grouped_properties=["title", "overview"]  # Optional parameter; for reducing prompt length
)

# Inspect the response
for o in response.objects:
    print(o.properties["title"])  # Print the title
print(response.generated)  # Print the generated text (the commonalities between them)

client.close()

In Time
Her
Gattaca
I, Robot
Children of Men
The movies listed share several common themes and elements:

1. **Science Fiction Genre**: All the movies fall under the science fiction genre, as indicated by the presence of genre ID **878** in their genre lists. They explore futuristic or speculative concepts.

2. **Futuristic Settings**: Each film is set in a future society, often with advanced technology, altered societal structures, or dystopian elements. For example:
   - *"In Time"* explores a future where time is currency.
   - *"Her"* depicts a future where AI and operating systems have advanced to the point of forming emotional relationships with humans.
   - *"Gattaca"* is set in a future where eugenics and genetic engineering play a central role in society.
   - *"I, Robot"* takes place in a world where robots are commonplace and governed by the Three Laws of Robotics.
   - *"Children of Men"* is set in a dystopian future where humans can no longer procreate.

3. **Social Commen